# 03 — MARL Training (Google Colab)

Trains price agent and sentiment agent for 500,000 timesteps each.

**Runtime → Change runtime type → T4 GPU**

## Step 1 — Check GPU

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/MARL/results/checkpoints", exist_ok=True)
os.makedirs("/content/drive/MyDrive/MARL/results/logs", exist_ok=True)
print("Drive mounted.")

## Step 3 — Clone repo and install dependencies
Replace yourusername with your GitHub username.

In [ ]:
!git clone https://github.com/yourusername/MARL.git
%cd MARL
!pip install -r requirements.txt -q
print("Dependencies installed.")

## Step 4 — Copy data files from Drive

In [ ]:
import shutil

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/sentiment", exist_ok=True)
os.makedirs("results/checkpoints", exist_ok=True)
os.makedirs("results/logs", exist_ok=True)

tickers = ["AAPL", "AMZN", "MSFT", "NVDA", "TSLA"]

shutil.copy("/content/drive/MyDrive/MARL/data/raw/prices_all.csv",
            "data/raw/prices_all.csv")
print("Copied prices_all.csv")

for t in tickers:
    shutil.copy(f"/content/drive/MyDrive/MARL/data/processed/{t}_processed.csv",
                f"data/processed/{t}_processed.csv")
    print(f"Copied {t}_processed.csv")

for t in tickers:
    src = f"/content/drive/MyDrive/MARL/data/sentiment/{t}_sentiment.csv"
    if os.path.exists(src):
        shutil.copy(src, f"data/sentiment/{t}_sentiment.csv")
        print(f"Copied {t}_sentiment.csv")
    else:
        print(f"WARNING: {t}_sentiment.csv not found in Drive")

## Step 5 — Add .env file with Gemini API key
Replace YOUR_GEMINI_KEY with your actual key.

In [ ]:
with open(".env", "w") as f:
    f.write("GEMINI_API_KEY=YOUR_GEMINI_KEY_HERE\n")
print(".env file created.")

## Step 6 — Validate environment

In [ ]:
import sys
sys.path.insert(0, "/content/MARL")

from src.training.train_marl import load_config
from src.environment.stock_env import StockTradingEnv

config = load_config()
env    = StockTradingEnv(config, mode="train")
print(f"Environment ready — {env.n_steps} trading days")
print(f"Agents: {env.possible_agents}")

## Step 7 — Run full training
Expect ~60-90 minutes on T4 GPU for 500,000 timesteps per agent.

In [ ]:
from src.training.train_marl import train
price_agent, sentiment_agent = train(config)
print("Training complete.")

## Step 8 — Copy checkpoints to Drive

In [ ]:
files_to_save = [
    "results/checkpoints/price_agent_final.zip",
    "results/checkpoints/sentiment_agent_final.zip",
]

for f in files_to_save:
    if os.path.exists(f):
        dst = f"/content/drive/MyDrive/MARL/{f}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(f, dst)
        print(f"Saved {f} to Drive")
    else:
        print(f"WARNING: {f} not found")

## Step 9 — Run baselines

In [ ]:
from src.training.train_baseline import run_all_baselines

summary = run_all_baselines(
    config,
    price_ckpt     = "results/checkpoints/price_agent_final.zip",
    sentiment_ckpt = "results/checkpoints/sentiment_agent_final.zip"
)
print(summary)

## Step 10 — Save all results to Drive

In [ ]:
result_files = [
    "results/logs/baseline_results.csv",
    "results/logs/baseline_summary.csv",
]

for f in result_files:
    if os.path.exists(f):
        dst = f"/content/drive/MyDrive/MARL/{f}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(f, dst)
        print(f"Saved {f} to Drive")

print("All results saved to Google Drive.")
print("Download and place in your local results/ folder.")